# How to Read Streaming Data From Files Using Kappa Architecture in Databricks
* Topic: Introduction to How to Read Streaming Data From Files Using Kappa Architecture in Databricks
* Author: Oindrila Chakraborty

# Objective
* 1. Read <b>Streaming</b> data from <b>JSON Files</b> that contains employee assignments related data, and, flatten the data in tabular format.
* 2. Write the output in <b>CSV</b> format.
* 3. Write the <b>Batch Code</b> first, and, then convert the <b>Batch Code</b> into <b>Streaming Code</b>.

# Sample Data

In [0]:
{
    "event_id": "1895604u-v56j-2504-a6b7-12hgtm0941",
    "event_offset": 10001,
    "event_publisher": "hr_system",
    "employee_id": "E1001",
    "data": {
        "assignments": [
            {
                "assignment_id": "A001",
                "start_from": "1st Jan 2024",
                "end_at": "30th Jun 2024",
                "status": "COMPLETED"
            },
            {
                "assignment_id": "A002",
                "start_from": "1st Jul 2024",
                "end_at": "31st Dec 2024",
                "status": "COMPLETED"
            },
            {
                "assignment_id": "A003",
                "start_from": "1st Jan 2025",
                "end_at": "31st Jul 2025",
                "status": "COMPLETED"
            },
            {
                "assignment_id": "A004",
                "start_from": "1st Aug 2025",
                "end_at": "31st Jan 2026",
                "status": "COMPLETED"
            },
            {
                "assignment_id": "A005",
                "start_from": "1st Feb 2026",
                "end_at": "30th Jun 2026",
                "status": "COMPLETED"
            },
            {
                "assignment_id": "A006",
                "start_from": "1st Jul 2026",
                "end_to": null,
                "status": "ACTIVE"
            }
        ]
    },
    "event_time": "2026-07-02 05:02:34.560325"
}

* The <b>JSON</b> file would contain certain fields, like -
    * <b>event_id</b>
    * <b>event_offset</b>
    * <b>event_publisher</b>
    * <b>employee_id</b>
* The field <b>assignments</b> is in the form of a <b>List</b> / <b>Array</b>, which contains the data for each of the assignments of the employee in each of the elements of the <b>List</b> / <b>Array</b>.
<br>Each element of the <b>List</b> / <b>Array</b> contains the below fields -
    * <b>assignment_id</b>
    * <b>start_from</b>
    * <b>end_at</b>
    * <b>status</b>
* The <b>JSON</b> file also contains a field <b>event_time</b>, which is there for each payload that is consumed as file.
* The target is to flatten the content of the <b>JSON</b> file into a tabular structure.
<br>After flattening out the content of the <b>JSON</b> file into a tabular structure, the fields of the <b>JSON</b> file will be present in the table in form of columns.
* The flattened table will have the below columns -
    * <b>event_id</b>
    * <b>event_offset</b>
    * <b>event_publisher</b>
    * <b>employee_id</b>
    * <b>assignment_id</b>
    * <b>start_from</b>
    * <b>end_at</b>
    * <b>status</b>
    * <b>event_time</b>

# Batch Format Code

In [0]:
from pyspark.sql.functions import *

#### Read Data from JSON File
* To read data in the <b>Batch</b> format, the data is kept and read from a file, placed in an input directory.
* Read the data as a <b>DataFrame</b>, named <b>streaming_df</b>.

In [0]:
streaming_df = (
    spark.read
         .format("json")
         .option("multiLine", True)
         .load("/Volumes/oc_catalog/oc_schema/oc_volume/input_files/")
)

#### View the Schema of the DataFrame Containing the Data of the JSON File

In [0]:
streaming_df.printSchema()

* It can be seen that <b>Spark</b> has successfully passed all the fields present in the <b>JSON</b> file.
* The <b>assignments</b> field is in the form of <b>List</b> / <b>Array</b> inside the payload. So, this needs to be exploded first using the <b>explode ()</b> function.
* The <b>Array</b> field, i.e., <b>assignments</b> is present inside the <b>data</b> field, which is of <b>Struct</b> type.
<br>That means, the field <b>data</b> is a <b>Struct</b> field, and, under this <b>Struct</b> field, the field <b>assignments</b> is present, which is an <b>Array</b> field.
<br>So, the <b>Array</b> field, i.e., <b>assignments</b> needs to be exploded.

#### Display the DataFrame Containing the Data of the JSON File

In [0]:
streaming_df.display()

#### Explode the "assignments" Column, As It Contains List / Array of Different Assignments of the Employee

In [0]:
exploded_df = streaming_df.withColumn("data_assignments", explode("data.assignments"))

#### View the Schema of the Exploded DataFrame

In [0]:
exploded_df.printSchema()

#### Display the Exploded DataFrame

In [0]:
exploded_df.display()

#### Flatten the Exploded DataFrame

* Since, the <b>JSON</b> payload contains two objects, the <b>streaming_df</b> has been exploded into two rows.
* The exploded data is present in the colummn <b>data_assignments</b>.
* Since, the data is exploded, the original column <b>data</b> is no longer required. So, this column can be dropped now.
* The exploded colummn <b>data_assignments</b> is a <b>Struct</b> field.
<br>In order to get the nested fields present in a <b>Struct</b> field, the <b>dot notation</b> is used.
<br>So, to expand the <b>Struct</b> field, i.e., <b>data_assignments</b> into multiple columns, with the same name as the nested fields, to flatten it out, the <b>dot notation</b> needs to be used.

In [0]:
flattened_df = (
    exploded_df
        .drop("data")
        .withColumn("assignment_id", col("data_assignments.assignment_id"))
        .withColumn("start_from", col("data_assignments.start_from"))
        .withColumn("end_at", col("data_assignments.end_at"))
        .withColumn("status", col("data_assignments.status"))
)

#### View the Schema of the Flattened DataFrame

In [0]:
flattened_df.printSchema()

#### Display the Flattened DataFrame

In [0]:
flattened_df.display()

#### Drop the Unnecessary Columns

* Since, the <b>flattened_df</b> contains the flattened data into four different columns, there is no need to keep the exploded column, i.e., <b>data_assignments</b> any longer.
<br>So, drop the column <b>data_assignments</b> in the final DataFrame, i.e., <b>final_df</b>.

In [0]:
final_df = flattened_df.drop("data_assignments")

#### View the Schema of the Final DataFrame

In [0]:
final_df.printSchema()

#### Display the Final DataFrame

In [0]:
final_df.display()

# Issue With Schema Inference While Reading Streaming Data Using Spark Structured Streaming in Databricks
* By default, <b>Spark Structured Streaming</b> requires an explicitly defined schema for file-based sources, like - <b>JSON</b>, <b>CSV</b>, and, <b>Text</b>, to guarantee that the stream is stable and consistent.
* However, in ad-hoc analysis, or, development use cases, it can be made sure that while reading streaming data <b>Spark</b> infers, or, reads the <b>Schema</b> from the <b>Streaming</b> data itself at run time, by setting the below configuration in the current <b>SparkSession</b> -
<br><b>spark.conf.set("spark.sql.streaming.schemaInference", "true")</b>.
* But, the above configuration often fails or behaves unexpectedly in production-grade <b>Databricks</b> environments due to the below reasons -
    * <b>Incorrect Configuration Scope</b>: Setting this option via <b>spark.conf.set()</b> inside a <b>Notebook</b> after the <b>Streaming Session</b> has initiated often fails to register with the active <b>Streaming Execution Context</b>.
    * <b>Checkpoint Interference</b>: : If the <b>Stream</b> uses a <b>Checkpoint</b> directory that has already captured an initial or empty <b>Schema</b>, then <b>Spark</b> will prioritize the <b>Checkpointed Schema</b> and ignore any <b>Schema Inference Configurations</b>.
    * <b>Format Limitations</b>: This configuration is only applicable to vanilla <b>File-Based Streaming Sources</b>, like - standard <b>JSON</b>, or, <b>CSV</b>.
    <br>It does not work on <b>Message Queues</b>, like - <b>Kafka</b>, or, <b>Kinesis</b>, nor does it work if the file is missing key path definitions.

# Solution of the Issue With Schema Inference While Reading Streaming Data Using Spark Structured Streaming in Databricks

## 1. Properly Apply the Spark Configuration Globally
* In a situation, where the <b>Spark Structured Streaming</b> must be used for file-based sources, the configuration <b>"spark.sql.streaming.schemaInference"</b> must be set globally before launching the <b>Stream</b>.
* It must be ensured that the configuration is set directly via the <b>SparkSession</b> object as below -
    * <b>spark.conf.set("spark.sql.streaming.schemaInference", "true")</b>
* Finally, the content of the <b>Checkpoint</b> directory must be cleared to build a fresh <b>Schema</b>.

In [0]:
from pyspark.sql.functions import *

# STEP 1: Set the Configuration at the First Cell of the Notebook
spark.conf.set("spark.sql.streaming.schemaInference", "true")

# STEP 2: Clear the Old Checkpoint Location to Ensure a Fresh Schema Build
dbutils.fs.rm("/path/to/checkpoint-directory/", True)

# STEP 3: Streaming Query
df = (spark.readStream
           .format("json")
           .load("/path/to/input/data/")
)

## 2. Switch to Databricks Auto Loader
* For the scenarios, where data is landing in <b>Cloud Object Storage</b>, like - <b>S3 Bucket</b>, <b>ADLS Container</b>, <b>GCS</b>, the <b>AutoLoader</b> should be used by defining the format as <b>cloudFiles</b>.
* <b>AutoLoader</b> is highly optimized for <b>Databricks</b> as it natively infers the <b>Schemas</b>, handles <b>Schema Evolution</b> seamlessly, and does not rely on the volatile configuration <b>"spark.sql.streaming.schemaInference"</b>.

In [0]:
# AutoLoader Handles Schema Inference and Schema Evolution Automatically
df = (spark.readStream
           .format("cloudFiles")
           .option("cloudFiles.format", "json") # csv, parquet, text, etc.
           .option("cloudFiles.schemaLocation", "/path/to/schema/metadata") # Required for Schema Inference
           .load("/path/to/input/data/")
)

# Clean Source to Archive the Read File Using Spark Structured Streaming
* As and when <b>Spark</b> processes, or, reads a file, it is possible to instruct <b>Spark</b> to perform a cleaning process at the landing zone, i.e., either to leave the processed, or, read file at the landing zone as it is, or, delete the file from landing zone, or, move the file to another location.
<br>What <b>Spark</b> should do after processing, or, reading a file is mentioned in the option <b>cleanSource</b>, in the <b>Read Stream</b> code.
* The <b>cleanSource</b> can take any one of the below three options -
    * <b>1</b>. <b>off</b>: The default option is <b>OFF</b>. When no option is specified for <b>cleanSource</b>, <b>Spark</b> will do nothing.
    <br>Input files will still be lying in the input directory after <b>Spark</b> finishes processing, or, reading the file.
    * <b>2</b>. <b>delete</b>: If the <b>delete</b> option is specified for <b>cleanSource</b>, <b>Spark</b> will delete the file from the input directory as soon as it processes, or, reads a file.
    <br>In production, the input file is not deleted after being processed, or, read.
    * <b>3</b>. <b>archive</b>: If the <b>archive</b> option is specified for <b>cleanSource</b>, <b>Spark</b> will move the file from the input directory as soon as it processes, or, reads a file, to the archive directory.
    <br>In production, the input file is archived into a location.
        * When <b>cleanSource</b> takes the option <b>archive</b>, it is also required to provide the directory location, where <b>Spark</b> will move the file once it processes, or, reads the file.
        <br>To do this, another option is provided as below -
        <br><b>option("sourceAchiveDir", "`directory-to-archive`")</b>

#### Challenges of Cleaning Source to Archive Read File in Databricks
* The options <b>cleanSource</b> and <b>sourceArchiveDir</b> are typically associated with <b>Spark Structured Streaming</b> for file-based sources like <b>CSV</b> or <b>JSON</b>.
* However, in Databricks, these options often fail or are ignored because Databricks uses its optimized <b>Auto Loader</b> (<b>cloudFiles</b>) connector, which uses a slightly different syntax.
    * <b>Auto Loader</b> Archival Option: <b>.option("cloudFiles.cleanSource", "MOVE")</b>
    * <b>Auto Loader</b> Archive Directory Option: <b>.option("cloudFiles.sourceArchiveDir", path)</b>

#### Decide Max Files to Read In Each Micro Batch
* To make sure that <b>Spark</b> reads only that many number of files from the input directory at each <b>Micro-Batch</b>, as specified in the <b>Read Stream</b> code, the option <b>maxFilesPerTrigger</b> is used.
* If <b>1</b> is provided as the value for the option <b>maxFilesPerTrigger</b>, then at each <b>Micro-Batch</b> run, <b>Spark</b> will only consume one file from the input directory.
* If this option is not provided altogether in the <b>Read Stream</b> code, then <b>Spark</b> will try to consume as many as files are present in the input directory.
* In production, in order to work the streaming process perfectly, this option is required to be provided.

# Convert the Batch Code into Streaming Code

#### Read Data from JSON File

In [0]:
from datetime import datetime

current_year = datetime.now().strftime('%Y')
current_month = datetime.now().strftime('%m')
current_day = datetime.now().strftime('%d')

streaming_df = (
    spark.readStream
         .format("json")
         .schema("employee_id STRING, data STRUCT<assignments: ARRAY<STRUCT<assignment_id: STRING, start_from: STRING, end_at: STRING, status: STRING>>>, event_id STRING, event_offset BIGINT, event_publisher STRING, event_time STRING")
         .option("multiLine", True)
        #  .option("cleanSource", "archive")
        #  .option("sourceArchiveDir", f"/Volumes/oc_catalog/oc_schema/oc_volume/archive/{current_year}/{current_month}/{current_day}/")
        #  .option("maxFilesPerTrigger", 1)
         .load("/Volumes/oc_catalog/oc_schema/oc_volume/input_files/")
)

#### View the Schema of the DataFrame Containing the Data of the JSON File

* In Databricks, if no <b>Schema</b> is explicitly specified and then it is tried to display the schema of a <b>Streaming DataFrame</b>, the below error is encountered -
<br><b>Schema must be specified when creating a streaming source DataFrame. If some files already exist in the directory, then depending on the file format you may be able to create a static DataFrame on that directory with 'spark.read.load(directory)' and infer schema from it.</b>
* So, it is mandatory to provide an explicit <b>Schema</b>.

In [0]:
streaming_df.printSchema()

#### Explode the "assignments" Column, As It Contains List / Array of  Device Reading

In [0]:
exploded_df = streaming_df.withColumn("data_assignments", explode("data.assignments"))

#### View the Schema of the Exploded DataFrame

In [0]:
exploded_df.printSchema()

#### Flatten the Exploded DataFrame

In [0]:
flattened_df = (
    exploded_df
        .drop("data")
        .withColumn("assignment_id", col("data_assignments.assignment_id"))
        .withColumn("start_from", col("data_assignments.start_from"))
        .withColumn("end_at", col("data_assignments.end_at"))
        .withColumn("status", col("data_assignments.status"))
)

#### View the Schema of the Flattened DataFrame

In [0]:
flattened_df.printSchema()

#### Drop the Unnecessary Columns

In [0]:
final_df = flattened_df.drop("data_assignments")

#### View the Schema of the Final DataFrame

In [0]:
final_df.printSchema()

#### Write the Output in CSV Format to Output Directory

* Once the <b>Streaming DataFrame</b> is ready, it can be written to the sink, i,e., into a <b>CSV</b> file in <b>append</b> mode.

#### Checkpoint directory
* The <b>Checkpoint directory</b> location stores the metadata information of a <b>Streaming Application</b>, such as - the offset, and, the files that are being processed, so that, even if the same file is put again in the input directory, <b>Spark</b> will not process that file.

#### Serverless Compute Restrictions on Running Spark Structured Streaming Code When No Trigger Mode is Mentioned
* If a <b>Streaming Query</b> is attempted to execute without an explicit trigger, then it defaults to the trigger type option <b>processingTime="0 seconds"</b>.
* If that <b>Streaming Query</b> is executed via a <b>Notebook</b>, or, a <b>Databricks Workflow</b> on a <b>Serverless Compute</b>, then that <b>Streaming Query</b> will fail with the error <b>INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED</b>.
* In this case, the trigger mode <b>availableNow</b> can be used to execute the <b>Streaming Query</b> successfully.

In [0]:
from datetime import datetime

current_year = datetime.now().strftime('%Y')
current_month = datetime.now().strftime('%m')
current_day = datetime.now().strftime('%d')

(
    final_df
        .writeStream
        .format("csv")
        .outputMode("append")
        .trigger(availableNow = True)
        .option("path", f"/Volumes/oc_catalog/oc_schema/oc_volume/output_files/{current_year}/{current_month}/{current_day}/")
        .option("checkpointLocation", "/Volumes/oc_catalog/oc_schema/oc_volume/checkpoint_location")
        .start()
        .awaitTermination()
)

# Different Types of Trigger Modes in Spark Structured Streaming

## 1. Once / AvailableNow
* Consider a case, where a <b>Streaming Workflow</b> containing the <b>Spark Streaming</b> code needs to be run only once, which will consume all the data, or, files that are available in the <b>Streaming</b> source, and, then the <b>Streaming Workflow</b> will be shut down.
<br>For this use case, the trigger mode <b>Once</b>, or, <b>AvailableNow</b> should be used.
* In the trigger mode <b>Once</b>, or, <b>AvailableNow</b>, the <b>Streaming Workflow</b> will behave like a <b>Batch Workflow</b>.
<br>This is one of the example how <b>Kappa Architecture</b> works.
* The trigger mode <b>Once</b>, or, <b>AvailableNow</b> will still read the data, or, files that are available in the <b>Streaming</b> source in incremental mode, but, will not trigger continuously in <b>Micro-Batches</b>, rather, will process all the data, or, files that are available in the <b>Streaming</b> source at that time in a <b>Single Batch</b> and will shut the <b>Stream</b> down.
    * In newer version of <b>Spark</b>, the trigger mode <b>AvailableNow</b> is found.
    * In previous version of <b>Spark</b>, the trigger mode <b>Once</b> were used. 

In [0]:
from datetime import datetime

current_year = datetime.now().strftime('%Y')
current_month = datetime.now().strftime('%m')
current_day = datetime.now().strftime('%d')

(
    final_df
        .writeStream
        .format("csv")
        .outputMode("append")
        .trigger(once = True)
        .option("path", f"/Volumes/oc_catalog/oc_schema/oc_volume/output_files/{current_year}/{current_month}/{current_day}/")
        .option("checkpointLocation", "/Volumes/oc_catalog/oc_schema/oc_volume/checkpoint_location/once_trigger/")
        .start()
        .awaitTermination()
)

In [0]:
from datetime import datetime

current_year = datetime.now().strftime('%Y')
current_month = datetime.now().strftime('%m')
current_day = datetime.now().strftime('%d')

(
    final_df
        .writeStream
        .format("csv")
        .outputMode("append")
        .trigger(availableNow = True)
        .option("path", f"/Volumes/oc_catalog/oc_schema/oc_volume/output_files/{current_year}/{current_month}/{current_day}/")
        .option("checkpointLocation", "/Volumes/oc_catalog/oc_schema/oc_volume/checkpoint_location/available_now_trigger/")
        .start()
        .awaitTermination()
)

## 2. processingTime
* In the trigger mode <b>processingTime</b>, <b>Spark Structured Streaming</b> will basically run the <b>Streaming Workflow</b> in <b>Micro-Batch</b> format at the specified time interval. 
* So, <b>.trigger(processingTime = "10 seconds")</b> informs <b>Spark Structured Streaming</b> that it has to run the <b>Streaming Workflow</b> in <b>Micro-Batch</b> format, meaning that <b>Spark Structured Streaming</b> has to process the data, or, the files in the <b>Streaming</b> source in <b>Incremental</b> fashion in every <b>10 Seconds</b>.
<br>So, whatever new data, or, files get accumulated in the <b>Streaming</b> source within the <b>10 Seconds</b>, <b>Spark Structured Streaming</b> will process those data, or, files, and, then will shut the <b>Streaming Workflow</b> down.
* <b>Spark Structured Streaming</b> will again re-run the <b>Streaming Workflow</b> after <b>10 Seconds</b> to process the newly accumulated data, or, files in the <b>Streaming</b> source.
<br>This will keep happening in every <b>10 Seconds</b> interval.
* This is how <b>Spark Structured Streaming</b> processes the <b>Streaming</b> data, or, files form the <b>Streaming</b> source in <b>Micro-Batch</b> format using the trigger mode <b>processingTime</b>.

In [0]:
from datetime import datetime

current_year = datetime.now().strftime('%Y')
current_month = datetime.now().strftime('%m')
current_day = datetime.now().strftime('%d')

(
    final_df
        .writeStream
        .format("csv")
        .outputMode("append")
        .trigger(processingTime = "10 seconds")
        .option("path", f"/Volumes/oc_catalog/oc_schema/oc_volume/output_files/{current_year}/{current_month}/{current_day}/")
        .option("checkpointLocation", "/Volumes/oc_catalog/oc_schema/oc_volume/checkpoint_location/processing_time_trigger/")
        .start()
        .awaitTermination()
)

## 3. continuous
* The trigger mode <b>continuous</b> is still in experimental mode, which does not process the data, or, files from the <b>Streaming</b> source in <b>Micro-Batches</b>.
<br>Instead, this trigger mode usually runs a <b>Streaming Workflow</b> with very low latency, i.e., this trigger mode will process the data continuously.
<br>Hence, this trigger mode is very fast.
* The time specified in this trigger mode, say <b>1 second</b>, is used to put the read data in <b>Checkpoint</b> directory.
<br>This means that in every <b>1 second</b>, the <b>continuous</b> mode will update its <b>Checkpoint</b> directory.
* This trigger mode is not supported by all the <b>Streamimg Sources</b>, and, <b>Streamimg Sinks</b>.
* The trigger mode <b>continuous</b> has some restrictions on execution of some of the queries, like -
    * <b>explode ()</b> function can not be run with it.
    * <b>Complex Aggregation Operations</b>, like - <b>Windowed Aggregations</b>, <b>Joins</b> are strictly unsupported.
    * <b>current_timestamp ()</b>, or <b>current_date ()</b> functions can not be run with it.
* The trigger mode <b>continuous</b> only allows -
    * Simple <b>Projections</b>, like - <b>select</b>, <b>map</b> functions.
    * Filter selections, like - <b>filter</b>, <b>where</b> functions.

In [0]:
from datetime import datetime

current_year = datetime.now().strftime('%Y')
current_month = datetime.now().strftime('%m')
current_day = datetime.now().strftime('%d')

(
    final_df
        .writeStream
        .format("csv")
        .outputMode("append")
        .trigger(continuous = "1 second")
        .option("path", f"/Volumes/oc_catalog/oc_schema/oc_volume/output_files/{current_year}/{current_month}/{current_day}/")
        .option("checkpointLocation", "/Volumes/oc_catalog/oc_schema/oc_volume/checkpoint_location/continuous_trigger/")
        .start()
        .awaitTermination()
)

# Unsupported Streaming Trigger Modes in Serverless Cluster Environment 
* Even the <b>Enterprise-Level</b>, <b>Production-Grade</b> <b>Serverless Compute Clusters</b> do not support the <b>processingTime</b>, and, <b>continuous</b> trigger modes.
<br>This restriction is a foundational limitation of <b>Databricks Serverless Compute</b> across all tiers—including <b>Premium Enterprise Workspaces</b>.
* If a <b>Streaming Query</b> is attempted to execute by specifying a time interval in the <b>processingTime</b>, or, <b>continuous</b> trigger modes, like - <b>processingTime="10 seconds"</b>, or, <b>continuous="1 second"</b> via a <b>Notebook</b>, or, a <b>Databricks Workflow</b> on a <b>Serverless Compute</b>, that <b>Streaming Query</b> will fail with the error <b>INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED</b>.

# Why Even Enterprise-Level, Production-Grade Serverless Compute Clusters Block Trigger Modes "processingTime" and "continuous"?
* Traditional time-based <b>Micro-Batches</b>, i.e., using the trigger mode <b>processingTime = "`<desired-interval-value>` seconds"</b>, or, continuous processing with very low latency, i.e., using the trigger mode <b>continuous = "`<desired-interval-value>` seconds/milliseconds"</b>, require a <b>Compute Cluster</b> to sit continuously idle in an "<b>always-on</b>" state while checking a <b>Streaming</b> source for the arrival of new data, or, files.
* This behavior conflicts directly with the architecture of <b>Serverless Compute Cluster</b>, which is optimized to instantly scale resources up to process a specific chunk of data and then immediately release the scaled up resources to minimize costs.

# Databricks Trigger Mode for Continuous Process - Real-Time Mode (RTM)
* <b>Real-Time Mode</b>, i.e., <b>RTM</b> is <b>Databricks</b>' replacement for the flawed, legacy trigger mode <b>continuous</b>.
* This trigger mode is built natively into <b>Apache Spark 4.1+</b>, and, fully integrated into the <b>Databricks Platform</b>.
* <b>RTM</b> brings end-to-end processing latencies down to <b>5 milliseconds</b> to <b>100 milliseconds</b>, which effectively provides a native alternative to <b>Apache Flink</b> without requiring to maintain a separate infrastructure stack.

# How Real-Time Mode (RTM) Works in Databricks?
* Standard <b>Micro-Batch</b> executes the <b>Stages</b> of a <b> Streaming Query</b> sequentially.
* In <b>Micro-Batch</b> execution, the <b>Spark Structured Streaming</b> -
    * Waits for an entire block of data, or, all arriving files to accumulate
    * Then process the accumulated data, or, the files
    * Finally, write the processed data to a <b>Checkpoint Log</b> before fetching the next block.
* <b>RTM</b> rewrites this lifecycle using the below three core architectural changes -
    * <b>1</b>. <b>Simultaneous Stage Scheduling</b>: Instead of scheduling the <b>Tasks</b> sequentially, <b>RTM</b> fires up all the <b>Stages</b> of the execution concurrently at the start of the query.
    * <b>2</b>. <b>Streaming Shuffles</b>: Data is passed downstream between the <b>Tasks</b> immediately <b>In-Memory</b> via a continuous <b>Shuffle</b> mechanism, rather than waiting for upstream <b>Map Stages</b> to completely write <b>Files</b> to <b>Disk</b>.
    * <b>3</b>. <b>Long-Running "Macro" Batches</b>: <b>RTM</b> groups the execution of a <b>Streaming Query</b> into massive, long-running chunks, defaulting to <b>5 Minutes</b>.
    <br>Within these <b>5 Minutes</b>, the <b>Streaming</b> of data gets completely unhindered, and, the output is instantly written to the <b>Sink</b>.
    <br>The <b>5 Minute</b> interval is simply the point at which the <b>Streaming Query</b> commits its global progress to the <b>Checkpoint Log</b>.
        * The "<b>5 Minutes</b>" argument tells <b>Spark Structured Streaming</b> how often to safely log the metadata of the <b>Streaming Query</b> in the <b>Checkpoint</b> directory to the underlying cloud storage layer.
        * Setting this value shorter, like "<b>1 Minute</b>" increases data recoverability precision, but, adds a small metadata logging overhead to the underlying cloud storage layer.

# Key Benefits of RTM Over the Legacy Continuous Trigger Mode
* Unlike the experimental <b>continuous</b> trigger mode, <b>RTM</b> supports full complex analytics workflows -
    * <b>Complex Aggregation Operations</b>, like - <b>Windowed Aggregations</b>, <b>Joins</b>, and, <b>transformWithState</b> are allowed.
    * <b>RTM</b> supports <b>Streaming-to-Static Joins</b>, or, <b>Streaming-to-Streaming Joins</b>.
    * <b>RTM</b> enables full support for <b>User-Defined Functions</b>, i.e., <b>UDF</b>s, and, standard <b>Spark SQL</b> functions, including - <b>explode ()</b>, <b>current_timestamp ()</b>, <b>current_date ()</b> etc.

# Use RTM on a Databricks Serverless Compute Cluster
* <b>Databricks Serverless Compute</b> eliminates the overhead of managing <b>VM</b> sizes, <b>Drivers</b>, and <b>Auto-Scaling</b> logic.
<br>To run ultra-low latency workloads on <b>Serverless Compute Cluster</b> using <b>RTM</b>, the standard <b>Structured Streaming</b> code can be written.
* <b>Step 1</b>. <b>Set Serverless Cluster Sizing Configuration</b>:
    * Because <b>RTM</b> schedules all the <b>Stages</b> of a <b>Streaming Query</b> concurrently, the <b>Serverless Compute Cluster</b> must have enough <b>Parallel Task Slots</b> to run all the <b>Stages</b> at the exact same time.
    * If the <b>Serverless Compute Cluster</b> does not have enough <b>Slots</b>, the <b>Streaming Workflow</b> will stall.
    * So, it must be ensured that the <b>Max Scaling</b> configuration of the <b>Serverless Compute Cluster</b> provides ample capacity for <b>Parallel Stage</b> execution.
* <b>Step 2</b>. <b>Write the Streaming Code using Trigger.RealTime</b>:
    * The standard <b>processingTime</b> trigger mode should be replaced by the <b>RTM</b> trigger mode, i.e., <b>.trigger(realTime=...)</b>.

In [0]:
# STEP 1: Read Messages from an Ultra-Low Latency Source, i.e., Apache Kafka Topic
streaming_df = (spark.readStream
                    .format("kafka")
                    .option("kafka.bootstrap.servers", "<BROKER_IP>:<PORT>")
                    .option("subscribe", "topic_name")
                    .load()
)

# STEP 2: Parse Message Values
parsed_stream_df = (streaming_df
                            .selectExpr("CAST(value AS STRING) AS data")
                            .select("data.*")
)

# STEP 3: Write to a Delta Table Using Real-Time Trigger
stream_query = (parsed_stream_df.writeStream
                              .format("delta")
                              .outputMode("append")
                              .trigger(realTime = "5 minutes")
                              .option("checkpointLocation", "/Volumes/oc_catalog/oc_schema/oc_volume/checkpoint_locations/real_time_data")
                              .toTable("oc_catalog.oc_schema.oc_real_time_table")
)